# Bacterial Assembly Results


In [ ]:
# import necessary modules
import os
import glob
import pandas as pd
import seaborn as sns; sns.set()
import matplotlib.pyplot as plt
import numpy as np

sns.set_style("ticks",{'axes.grid' : True})
sns.set_palette("colorblind")

plt.rcParams["axes.linewidth"] = 1.5
plt.rcParams["xtick.major.width"] = 1.5
plt.rcParams["ytick.major.width"] = 1.5
plt.rcParams["xtick.major.size"] = 8
plt.rcParams["ytick.major.size"] = 8
plt.rcParams["axes.titlepad"] = 20

plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams["axes.titlesize"] = 30
plt.rcParams['axes.labelsize'] = 23.5
plt.rcParams['xtick.labelsize'] = 18
plt.rcParams['ytick.labelsize'] = 18
plt.rcParams['legend.fontsize'] = 18
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Liberation Sans']
plt.rcParams['text.usetex'] = False
plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams["savefig.dpi"]=300


In [ ]:
def as_list(x):
	if isinstance(x, str):
		return([x])
	return(list(x))

def get_named(container, name, default=None):
	try:
		return(getattr(container, name))
	except Exception:
		return(default)

input_checkm=as_list(snakemake.input.checkm)
input_sourmash=as_list(snakemake.input.sourmash)
input_gtdbtk=as_list(snakemake.input.gtdbtk)
input_quast=snakemake.input.quast
input_coverage=as_list(snakemake.input.coverage)
input_genomad=as_list(get_named(snakemake.input, "genomad", []))

SAMPLING=snakemake.params.sampling
LONG_ASSEMBLER=snakemake.params.long_assembler
LONG_ASSEMBLER_PACBIO=snakemake.params.long_assembler_pacbio

output_summary_html=snakemake.output.summary_html
output_summary_csv=snakemake.output.summary_csv
output_checkm_png=snakemake.output.checkm_png
output_checkm_svg=snakemake.output.checkm_svg
output_taxonomy_png=snakemake.output.taxonomy_png
output_taxonomy_svg=snakemake.output.taxonomy_svg
output_quality_taxonomy_png=snakemake.output.quality_taxonomy_png
output_quality_taxonomy_svg=snakemake.output.quality_taxonomy_svg
output_genomad_counts_png=get_named(snakemake.output, "genomad_counts_png", None)
output_genomad_counts_svg=get_named(snakemake.output, "genomad_counts_svg", None)
output_genomad_lengths_png=get_named(snakemake.output, "genomad_lengths_png", None)
output_genomad_lengths_svg=get_named(snakemake.output, "genomad_lengths_svg", None)


In [ ]:
def clean_sample_name(sample):
	sample=str(sample)
	sample=os.path.basename(sample)
	sample=sample.replace(".fasta", "")
	sample=sample.replace(".fa", "")
	sample=sample.replace(".fna", "")
	sample=sample.replace("racon_", "")
	sample=sample.replace("polypolish_", "")
	sample=sample.replace("_spades_filtered_scaffolds." + SAMPLING, "")
	sample=sample.replace("_contigs_2_" + LONG_ASSEMBLER + "." + SAMPLING, "")
	sample=sample.replace("_contigs_" + LONG_ASSEMBLER + "." + SAMPLING, "")
	sample=sample.replace("_contigs_" + LONG_ASSEMBLER_PACBIO + "." + SAMPLING, "")
	sample=sample.replace("_" + LONG_ASSEMBLER + "_corrected_scaffolds_pilon." + SAMPLING, "")
	sample=sample.replace("_" + SAMPLING + "_nanopore.classifications.csv", "")
	sample=sample.replace("_" + SAMPLING + "_nanopore_hybrid.classifications.csv", "")
	sample=sample.replace("_" + SAMPLING + "_pacbio.classifications.csv", "")
	sample=sample.replace("_" + SAMPLING + "_pacbio_hybrid.classifications.csv", "")
	sample=sample.replace("_" + SAMPLING + ".classifications.csv", "")
	sample=sample.replace("_checkM_" + SAMPLING, "")
	sample=sample.replace("_long_geNomad_" + SAMPLING, "")
	sample=sample.replace("_pacbio_geNomad_" + SAMPLING, "")
	sample=sample.replace("_geNomad_" + SAMPLING, "")
	return(sample)


## Parse CheckM


In [ ]:
def parse_checkm_dir(path):
	sample=clean_sample_name(os.path.basename(path))
	out={"sample":sample, "checkm_completeness":np.nan, "checkm_contamination":np.nan, "checkm_strain_heterogeneity":np.nan, "checkm_marker_lineage":np.nan, "checkm_genome_size":np.nan}
	candidates=[]
	candidates.extend(glob.glob(path + "/*storage/bin_stats_ext.tsv"))
	candidates.extend(glob.glob(path + "/storage/bin_stats_ext.tsv"))
	candidates.extend(glob.glob(path + "/*quality_report.tsv"))
	candidates.extend(glob.glob(path + "/quality_report.tsv"))
	candidates.extend(glob.glob(path + "/*.tsv"))
	for candidate in candidates:
		try:
			df=pd.read_csv(candidate, sep="\t")
			cols=list(df.columns)
			if "Completeness" in cols and "Contamination" in cols:
				row=df.iloc[0]
				out["checkm_completeness"]=row.get("Completeness", np.nan)
				out["checkm_contamination"]=row.get("Contamination", np.nan)
				out["checkm_strain_heterogeneity"]=row.get("Strain heterogeneity", np.nan)
				out["checkm_marker_lineage"]=row.get("Marker lineage", np.nan)
				out["checkm_genome_size"]=row.get("Genome size", np.nan)
				return(out)
		except Exception:
			pass
	log_file=path + ".log"
	if os.path.exists(log_file):
		with open(log_file) as handle:
			lines=[line.rstrip("\n") for line in handle]
		for line in lines:
			if "Completeness" in line and "Contamination" in line and "Strain heterogeneity" in line:
				continue
			if line.strip().startswith("-"):
				continue
			if "UID" in line and len(line.split()) >= 15:
				parts=line.split()
				try:
					out["sample"]=clean_sample_name(parts[0])
					out["checkm_marker_lineage"]=" ".join(parts[1:-13])
					out["checkm_completeness"]=float(parts[-3])
					out["checkm_contamination"]=float(parts[-2])
					out["checkm_strain_heterogeneity"]=float(parts[-1])
					return(out)
				except Exception:
					pass
	return(out)

checkm_rows=[]
for path in input_checkm:
	checkm_rows.append(parse_checkm_dir(path))

checkm_df=pd.DataFrame(checkm_rows)
if len(checkm_df)==0:
	checkm_df=pd.DataFrame(columns=["sample", "checkm_completeness", "checkm_contamination", "checkm_strain_heterogeneity", "checkm_marker_lineage", "checkm_genome_size"])
for col in ["checkm_completeness", "checkm_contamination", "checkm_strain_heterogeneity", "checkm_genome_size"]:
	if col in checkm_df.columns:
		checkm_df[col]=pd.to_numeric(checkm_df[col], errors="coerce")
checkm_df


## Parse Sourmash classifications


In [ ]:
def parse_sourmash(path):
	sample=clean_sample_name(os.path.basename(path))
	out={"sample":sample, "sourmash_taxonomy":np.nan, "sourmash_superkingdom":np.nan, "sourmash_phylum":np.nan, "sourmash_class":np.nan, "sourmash_order":np.nan, "sourmash_family":np.nan, "sourmash_genus":np.nan, "sourmash_species":np.nan}
	try:
		df=pd.read_csv(path)
		if len(df)==0:
			return(out)
		if "lineage" in df.columns:
			lineage=str(df.iloc[0]["lineage"])
			out["sourmash_taxonomy"]=lineage
			lineage_parts=lineage.split(";")
			lineage_ranks=["sourmash_superkingdom", "sourmash_phylum", "sourmash_class", "sourmash_order", "sourmash_family", "sourmash_genus", "sourmash_species"]
			for rank, value in zip(lineage_ranks, lineage_parts):
				if value not in ["", "N/A", "nan"]:
					out[rank]=value
		if "rank" in df.columns and "name" in df.columns:
			for rank in ["superkingdom", "phylum", "class", "order", "family", "genus", "species"]:
				sub=df[df["rank"]==rank]
				if len(sub)>0:
					out["sourmash_" + rank]=sub.iloc[0]["name"]
		for col in df.columns:
			lcol=col.lower()
			if lcol in ["superkingdom", "phylum", "class", "order", "family", "genus", "species"]:
				out["sourmash_" + lcol]=df.iloc[0][col]
	except Exception:
		pass
	return(out)

sourmash_rows=[]
for path in input_sourmash:
	sourmash_rows.append(parse_sourmash(path))

sourmash_df=pd.DataFrame(sourmash_rows)
if len(sourmash_df)==0:
	sourmash_df=pd.DataFrame(columns=["sample", "sourmash_taxonomy", "sourmash_superkingdom", "sourmash_phylum", "sourmash_class", "sourmash_order", "sourmash_family", "sourmash_genus", "sourmash_species"])
sourmash_df


## Parse GTDB-Tk


In [ ]:
def parse_gtdbtk_dir(path):
	rows=[]
	files=[]
	files.extend(glob.glob(path + "/classify/gtdbtk.bac120.summary.tsv"))
	files.extend(glob.glob(path + "/gtdbtk.bac120.summary.tsv"))
	files.extend(glob.glob(path + "/**/gtdbtk.bac120.summary.tsv", recursive=True))
	for file in files:
		try:
			df=pd.read_csv(file, sep="\t")
			if "user_genome" in df.columns:
				for _, row in df.iterrows():
					sample=clean_sample_name(row["user_genome"])
					rows.append({
						"sample":sample,
						"gtdbtk_classification":row.get("classification", np.nan),
						"gtdbtk_fastani_reference":row.get("fastani_reference", np.nan),
						"gtdbtk_fastani_ani":row.get("fastani_ani", np.nan),
						"gtdbtk_fastani_af":row.get("fastani_af", np.nan),
						"gtdbtk_note":row.get("note", np.nan)
					})
		except Exception:
			pass
	return rows

gtdbtk_rows=[]
for path in input_gtdbtk:
	gtdbtk_rows.extend(parse_gtdbtk_dir(path))

gtdbtk_df=pd.DataFrame(gtdbtk_rows)
if len(gtdbtk_df)==0:
	gtdbtk_df=pd.DataFrame(columns=["sample", "gtdbtk_classification", "gtdbtk_fastani_reference", "gtdbtk_fastani_ani", "gtdbtk_fastani_af", "gtdbtk_note"])
for col in ["gtdbtk_fastani_ani", "gtdbtk_fastani_af"]:
	if col in gtdbtk_df.columns:
		gtdbtk_df[col]=pd.to_numeric(gtdbtk_df[col], errors="coerce")
gtdbtk_df=gtdbtk_df.drop_duplicates(subset=["sample"])
gtdbtk_df


## Parse QUAST


In [ ]:
quast_df=pd.read_csv(input_quast, sep="\t")

if "Assembly" not in quast_df.columns:
	quast_df=quast_df.rename(columns={quast_df.columns[0]: "Assembly"})

quast_df["sample"]=quast_df["Assembly"].astype(str).apply(clean_sample_name)

quast_cols=["sample", "# contigs (>= 1000 bp)", "Total length", "N50", "Largest contig", "GC (%)"]
quast_cols=[col for col in quast_cols if col in quast_df.columns]

quast_df=quast_df[quast_cols].copy()
quast_df=quast_df.rename(columns={
	"# contigs (>= 1000 bp)": "assembly_contigs_1000bp",
	"Total length": "assembly_total_length",
	"N50": "assembly_N50",
	"Largest contig": "assembly_largest_contig",
	"GC (%)": "assembly_GC"
})
quast_df


## Coverage


In [ ]:
def parse_coverage(path):
	sample=clean_sample_name(os.path.basename(path))
	df=pd.read_csv(path, sep="\t")
	df.columns=[col.replace(" ", "_").replace("/", "_").lower() for col in df.columns]
	out={"sample":sample}
	for col in df.columns:
		if "mean" in col:
			out["coverage_mean"]=pd.to_numeric(df[col], errors="coerce").mean()
		if "covered_bases" in col:
			out["coverage_covered_bases"]=pd.to_numeric(df[col], errors="coerce").sum()
		if "length" in col:
			out["coverage_total_length"]=pd.to_numeric(df[col], errors="coerce").sum()
		if "count" in col:
			out["coverage_read_count"]=pd.to_numeric(df[col], errors="coerce").sum()
		if "rpkm" in col:
			out["coverage_mean_rpkm"]=pd.to_numeric(df[col], errors="coerce").mean()
	if "coverage_covered_bases" in out and "coverage_total_length" in out and out["coverage_total_length"] != 0:
		out["coverage_breadth_percent"]=out["coverage_covered_bases"]*100/out["coverage_total_length"]
	return(out)

coverage_rows=[]
for path in input_coverage:
	coverage_rows.append(parse_coverage(path))

coverage_df=pd.DataFrame(coverage_rows)
if len(coverage_df)==0:
	coverage_df=pd.DataFrame(columns=["sample", "coverage_mean", "coverage_covered_bases", "coverage_total_length", "coverage_read_count", "coverage_mean_rpkm", "coverage_breadth_percent"])

coverage_df


## Parse geNomad plasmid/phage/prophage summaries


In [ ]:
def parse_genomad_dir(path):
	sample=clean_sample_name(os.path.basename(path))
	out={
		"sample":sample,
		"genomad_n_plasmids":0,
		"genomad_n_phages":0,
		"genomad_n_prophages":0,
		"genomad_plasmid_total_length":0,
		"genomad_phage_total_length":0,
		"genomad_prophage_total_length":0,
		"genomad_mobile_total_length":0,
		"genomad_plasmid_contigs":np.nan,
		"genomad_phage_contigs":np.nan,
		"genomad_prophage_contigs":np.nan
	}
	plasmid_files=glob.glob(path + "/**/*_plasmid_summary.tsv", recursive=True)
	virus_files=glob.glob(path + "/**/*_virus_summary.tsv", recursive=True)
	provirus_files=glob.glob(path + "/**/*_provirus_summary.tsv", recursive=True)
	plasmid_contigs=[]
	phage_contigs=[]
	prophage_contigs=[]

	for file in plasmid_files:
		try:
			df=pd.read_csv(file, sep="\t")
			out["genomad_n_plasmids"]+=len(df)
			if "length" in df.columns:
				out["genomad_plasmid_total_length"]+=pd.to_numeric(df["length"], errors="coerce").sum()
			if "seq_name" in df.columns:
				plasmid_contigs.extend(df["seq_name"].astype(str).tolist())
		except Exception:
			pass

	for file in virus_files:
		try:
			df=pd.read_csv(file, sep="\t")
			out["genomad_n_phages"]+=len(df)
			if "length" in df.columns:
				out["genomad_phage_total_length"]+=pd.to_numeric(df["length"], errors="coerce").sum()
			if "seq_name" in df.columns:
				phage_contigs.extend(df["seq_name"].astype(str).tolist())
		except Exception:
			pass

	for file in provirus_files:
		try:
			df=pd.read_csv(file, sep="\t")
			out["genomad_n_prophages"]+=len(df)
			if "length" in df.columns:
				out["genomad_prophage_total_length"]+=pd.to_numeric(df["length"], errors="coerce").sum()
			if "seq_name" in df.columns:
				prophage_contigs.extend(df["seq_name"].astype(str).tolist())
		except Exception:
			pass

	out["genomad_mobile_total_length"]=out["genomad_plasmid_total_length"]+out["genomad_phage_total_length"]+out["genomad_prophage_total_length"]
	out["genomad_plasmid_contigs"]=";".join(sorted(set(plasmid_contigs))) if len(plasmid_contigs)>0 else np.nan
	out["genomad_phage_contigs"]=";".join(sorted(set(phage_contigs))) if len(phage_contigs)>0 else np.nan
	out["genomad_prophage_contigs"]=";".join(sorted(set(prophage_contigs))) if len(prophage_contigs)>0 else np.nan
	return(out)

genomad_rows=[]
for path in input_genomad:
	genomad_rows.append(parse_genomad_dir(path))

genomad_df=pd.DataFrame(genomad_rows)
if len(genomad_df)==0:
	genomad_df=pd.DataFrame(columns=["sample", "genomad_n_plasmids", "genomad_n_phages", "genomad_n_prophages", "genomad_plasmid_total_length", "genomad_phage_total_length", "genomad_prophage_total_length", "genomad_mobile_total_length", "genomad_plasmid_contigs", "genomad_phage_contigs", "genomad_prophage_contigs"])

genomad_df


## Merge bacterial result tables


In [ ]:
summary_df=checkm_df.merge(quast_df, on="sample", how="outer").merge(coverage_df, on="sample", how="outer").merge(genomad_df, on="sample", how="outer").merge(sourmash_df, on="sample", how="outer").merge(gtdbtk_df, on="sample", how="outer")

def checkm_quality(row):
	if pd.isna(row.get("checkm_completeness", np.nan)) or pd.isna(row.get("checkm_contamination", np.nan)):
		return "Not available"
	if row["checkm_completeness"] >= 90 and row["checkm_contamination"] <= 5:
		return "High-quality"
	if row["checkm_completeness"] >= 50 and row["checkm_contamination"] <= 10:
		return "Medium-quality"
	return "Low-quality"

summary_df["checkm_quality"]=summary_df.apply(checkm_quality, axis=1)

if "gtdbtk_classification" in summary_df.columns:
	summary_df["gtdbtk_genus"]=summary_df["gtdbtk_classification"].astype(str).str.extract(r"g__([^;]*)")[0]
	summary_df["gtdbtk_species"]=summary_df["gtdbtk_classification"].astype(str).str.extract(r"s__([^;]*)")[0]

for col in ["genomad_mobile_total_length", "genomad_plasmid_total_length", "genomad_phage_total_length", "genomad_prophage_total_length"]:
	if col not in summary_df.columns:
		summary_df[col]=0
	summary_df[col]=pd.to_numeric(summary_df[col], errors="coerce").fillna(0)

if "assembly_total_length" in summary_df.columns:
	summary_df["assembly_non_mobile_length"]=summary_df["assembly_total_length"]-summary_df["genomad_mobile_total_length"]
	summary_df["assembly_non_mobile_length"]=summary_df["assembly_non_mobile_length"].clip(lower=0)
	summary_df["assembly_mobile_percent"]=summary_df["genomad_mobile_total_length"]*100/summary_df["assembly_total_length"]
	summary_df["assembly_non_mobile_percent"]=summary_df["assembly_non_mobile_length"]*100/summary_df["assembly_total_length"]

summary_df=summary_df.sort_values("sample")
summary_df.to_csv(output_summary_csv, index=False)

gradient_cols=[
	"assembly_contigs_1000bp", "assembly_total_length", "assembly_N50", "assembly_largest_contig", "assembly_GC",
	"assembly_non_mobile_length", "assembly_mobile_percent",
	"genomad_n_plasmids", "genomad_n_phages", "genomad_n_prophages", "genomad_mobile_total_length",
	"checkm_completeness", "checkm_contamination",
	"gtdbtk_fastani_ani", "gtdbtk_fastani_af",
	"coverage_mean", "coverage_breadth_percent", "coverage_read_count"
]
gradient_cols=[col for col in gradient_cols if col in summary_df.columns]
summary_df_out=summary_df.style.background_gradient(cmap="RdYlGn", subset=gradient_cols).to_html()
with open(output_summary_html, "w") as fp:
	fp.write(summary_df_out)

summary_df


## CheckM completeness and contamination


In [ ]:
plot_df=summary_df.dropna(subset=["checkm_completeness", "checkm_contamination"]).copy()
fig_width=max(8, 0.45*len(plot_df))
fig, ax = plt.subplots(figsize=(fig_width, 10))

plot_df=plot_df.melt(id_vars=["sample", "checkm_quality"], value_vars=["checkm_completeness", "checkm_contamination"], var_name="metric", value_name="percent")
plot_df["metric"]=plot_df["metric"].replace({"checkm_completeness":"Completeness", "checkm_contamination":"Contamination"})

sns.barplot(data=plot_df, x="sample", y="percent", hue="metric", ax=ax)
ax.axhline(90, linestyle="--", linewidth=1, alpha=0.5)
ax.axhline(5, linestyle=":", linewidth=1, alpha=0.5)
ax.set_xlabel("Sample")
ax.set_ylabel("Percent")
ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), title="CheckM metric")

fig.savefig(output_checkm_png, format="png", bbox_inches="tight", transparent=True)
fig.savefig(output_checkm_svg, format="svg", bbox_inches="tight", transparent=True)
plt.show()


## geNomad plasmid, phage and prophage counts


In [ ]:
plot_df=summary_df[["sample", "genomad_n_plasmids", "genomad_n_phages", "genomad_n_prophages"]].copy()
plot_df=plot_df.melt(id_vars="sample", var_name="element_type", value_name="count")
plot_df["element_type"]=plot_df["element_type"].replace({
	"genomad_n_plasmids":"Plasmids",
	"genomad_n_phages":"Phages",
	"genomad_n_prophages":"Prophages"
})

fig_width=max(8, 0.45*summary_df["sample"].nunique())
fig, ax = plt.subplots(figsize=(fig_width, 10))
sns.barplot(data=plot_df, x="sample", y="count", hue="element_type", ax=ax)
ax.set_xlabel("Sample")
ax.set_ylabel("Number of mobile elements")
ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), title="geNomad class")

if output_genomad_counts_png is not None:
	fig.savefig(output_genomad_counts_png, format="png", bbox_inches="tight", transparent=True)
if output_genomad_counts_svg is not None:
	fig.savefig(output_genomad_counts_svg, format="svg", bbox_inches="tight", transparent=True)

plt.show()


## Assembly length classified as plasmid, phage, prophage or non-mobile


In [ ]:
plot_df=summary_df[["sample", "genomad_plasmid_total_length", "genomad_phage_total_length", "genomad_prophage_total_length", "assembly_non_mobile_length"]].copy()
plot_df=plot_df.melt(id_vars="sample", var_name="fraction", value_name="length")
plot_df["fraction"]=plot_df["fraction"].replace({
	"genomad_plasmid_total_length":"Plasmid",
	"genomad_phage_total_length":"Phage",
	"genomad_prophage_total_length":"Prophage",
	"assembly_non_mobile_length":"Not plasmid/phage"
})
plot_df["length_Mbp"]=pd.to_numeric(plot_df["length"], errors="coerce")/1000000

fig_width=max(8, 0.45*summary_df["sample"].nunique())
fig, ax = plt.subplots(figsize=(fig_width, 10))
sns.barplot(data=plot_df, x="sample", y="length_Mbp", hue="fraction", ax=ax)
ax.set_xlabel("Sample")
ax.set_ylabel("Assembly length (Mbp)")
ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), title="")

if output_genomad_lengths_png is not None:
	fig.savefig(output_genomad_lengths_png, format="png", bbox_inches="tight", transparent=True)
if output_genomad_lengths_svg is not None:
	fig.savefig(output_genomad_lengths_svg, format="svg", bbox_inches="tight", transparent=True)

plt.show()


## Taxonomic overview


In [ ]:
tax_col="gtdbtk_genus"
if tax_col not in summary_df.columns or summary_df[tax_col].replace("nan", np.nan).dropna().empty:
	tax_col="sourmash_genus"

plot_df=summary_df.copy()
if tax_col in plot_df.columns:
	plot_df[tax_col]=plot_df[tax_col].replace("", np.nan).fillna("Unclassified")
	tax_counts=plot_df.groupby(tax_col).size().reset_index(name="count").sort_values("count", ascending=False)
	fig_width=max(8, 0.7*len(tax_counts))
	fig, ax = plt.subplots(figsize=(fig_width, 10))
	sns.barplot(data=tax_counts, x=tax_col, y="count", ax=ax)
	ax.set_xlabel(tax_col)
	ax.set_ylabel("Number of samples")
	ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
	fig.savefig(output_taxonomy_png, format="png", bbox_inches="tight", transparent=True)
	fig.savefig(output_taxonomy_svg, format="svg", bbox_inches="tight", transparent=True)
	plt.show()
else:
	print("No genus-level taxonomy column available.")


## Quality by taxonomy


In [ ]:
tax_col="gtdbtk_genus"
if tax_col not in summary_df.columns or summary_df[tax_col].replace("nan", np.nan).dropna().empty:
	tax_col="sourmash_genus"

plot_df=summary_df.copy()
if tax_col in plot_df.columns:
	plot_df[tax_col]=plot_df[tax_col].replace("", np.nan).fillna("Unclassified")
	fig_width=max(8, 0.7*plot_df[tax_col].nunique())
	fig, ax = plt.subplots(figsize=(fig_width, 10))
	sns.scatterplot(data=plot_df, x="checkm_completeness", y="checkm_contamination", hue=tax_col, style="checkm_quality", s=100, ax=ax)
	ax.axvline(90, linestyle="--", linewidth=1, alpha=0.5)
	ax.axhline(5, linestyle="--", linewidth=1, alpha=0.5)
	ax.set_xlabel("CheckM completeness (%)")
	ax.set_ylabel("CheckM contamination (%)")
	ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), title="Taxonomy")
	fig.savefig(output_quality_taxonomy_png, format="png", bbox_inches="tight", transparent=True)
	fig.savefig(output_quality_taxonomy_svg, format="svg", bbox_inches="tight", transparent=True)
	plt.show()
else:
	print("No taxonomy column available.")
